# Motor Cortex Dynamics — reproduction + interactive viewer
# 운동피질 동역학 — 재현 + 인터랙티브 뷰어

**Stage 1 (descriptive):** PCA + hand-implemented **jPCA** on real M1 data (MC_Maze) → rotational dynamics (Churchland et al. 2012).

**Stage 2 (mechanistic):** **fixed-point analysis** of a task-trained RNN → the local structure that *generates* the rotation (Sussillo & Barak 2013; Sussillo et al. 2015).

jPCA is *descriptive* — one global skew-symmetric linear fit to trajectory geometry, needs no equations, runs on **brain and RNN**. Fixed-point analysis is *mechanistic* — needs the evaluable vector field `dx/dt`, so it runs on the **RNN only**. Fixed points **generate** the rotation jPCA **describes**. The viewer keeps this asymmetry visible: brain = trajectories + jPCA plane; RNN = trajectories + jPCA + fixed points + flow field.

Each section follows: **(1)** markdown math + *why* → **(2)** code → **(3)** inline plotly → **(4)** markdown interpretation + failure modes.

---

**한국어 설명**

**1단계 (기술적/descriptive):** 실제 M1 데이터(MC_Maze)에 PCA + **직접 구현한 jPCA** 적용 → 회전 동역학 (Churchland et al. 2012).

**2단계 (기전적/mechanistic):** 과제로 학습한 RNN의 **고정점 분석(fixed-point analysis)** → 그 회전을 *생성하는* 국소 구조 (Sussillo & Barak 2013; Sussillo et al. 2015).

jPCA는 *기술적* 방법입니다 — 궤적 기하에 하나의 전역 반대칭(skew-symmetric) 선형계를 적합하며, 방정식이 필요 없으므로 **뇌와 RNN 모두**에 적용됩니다. 고정점 분석은 *기전적* 방법으로, 평가 가능한 벡터장 `dx/dt`가 필요하므로 **RNN에만** 적용됩니다. 고정점이 jPCA가 *기술하는* 회전을 *생성합니다*. 뷰어는 이 비대칭을 계속 드러냅니다: 뇌 = 궤적 + jPCA 평면 / RNN = 궤적 + jPCA + 고정점 + 흐름장(flow field).

각 절의 구성: **(1)** 마크다운 수식 + *이유* → **(2)** 코드 → **(3)** 인라인 plotly → **(4)** 마크다운 해석 + 관찰된 실패 모드.

## 0 · Environment setup / 환경 설정
Mount Drive first (Colab filesystem is ephemeral), then install deps and import the `python/` modules.

**한국어:** Colab 파일시스템은 휘발성이라 먼저 Google Drive를 마운트한 뒤, 의존성을 설치하고 `python/` 모듈을 임포트합니다.

In [ ]:
# --- FIRST CELL: mount Google Drive so downloads/weights persist across sessions ---
# 첫 셀: 다운로드/가중치를 세션 간에 유지하기 위해 Google Drive 마운트
# WHY / 이유: Colab 로컬 디스크는 런타임이 재활용될 때 지워진다. MC_Maze는 크고
# RNN 학습은 느리므로 둘 다 Drive에 캐시한다 -> 새 세션은 재다운로드/재학습 없이 재사용.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/neural-dynamics')
    IN_COLAB = True
except ModuleNotFoundError:
    # Running locally (not Colab): fall back to a repo-relative location.
    # 로컬 실행(비-Colab) 시: 저장소 기준 경로로 대체.
    PROJECT_ROOT = Path.cwd().resolve().parent
    IN_COLAB = False

CACHE_DIR = PROJECT_ROOT / 'cache'   # MC_Maze downloads + trained RNN weights / 다운로드+학습 가중치 캐시
DATA_DIR  = PROJECT_ROOT / 'data'    # exported web JSON (the 'plate' reads this) / 뷰어가 읽는 JSON
for d in (CACHE_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f'IN_COLAB={IN_COLAB}  CACHE_DIR={CACHE_DIR}  DATA_DIR={DATA_DIR}')

In [ ]:
# --- Dependencies (run once per session) / 의존성 (세션당 한 번 실행) ---
# Hand code is primary; the two git packages are differential-test ORACLES.
# 직접 구현이 기본 경로이고, 아래 git 패키지 2개는 차등 검증(differential-test)의 기준(oracle)이다.
# %pip install -q numpy scipy scikit-learn pandas torch h5py nlb_tools dandi plotly ipywidgets nbformat
# %pip install -q git+https://github.com/bantin/jPCA.git
# %pip install -q git+https://github.com/tripdancer0916/pytorch-fixed-point-analysis.git

In [ ]:
# --- Import the pipeline modules (logic lives in python/, not in cells) ---
# 파이프라인 모듈 임포트 (로직은 셀이 아니라 python/ 에 둔다)
import sys
REPO = PROJECT_ROOT if (PROJECT_ROOT / 'python').exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO))
# from python import data, pca_jpca, rnn, fixedpoints, flowfield, export  # (enabled as modules fill in / 모듈이 채워지면 활성화)
import plotly.io as pio
pio.renderers.default = 'colab' if IN_COLAB else 'notebook'
print('modules path:', REPO)

## M0 · jPCA hand implementation _(gated — awaiting approval)_
**(1) math/why:** soft-normalize → subtract cross-condition mean per timepoint (Lebedev 2019 critique noted, kept for faithful reproduction) → PCA (k=6) → finite-diff `Ẋ` → fit `Ẋ = M X` with `M = -Mᵀ` (skew-symmetric constrained least squares, closed form) → eigendecompose (±iω) → top plane.
**PASS/FAIL:** fit R² of `Ẋ=MX`; top rotation-plane variance fraction; principal angle vs Antin `jPCA` < 5°, freq within 5%.

---

## M0 · jPCA 직접 구현 _(게이트 — 승인 대기 중)_
**(1) 수식/이유:** 소프트 정규화 → 시점별 조건간 평균(cross-condition mean) 차감 (Lebedev 2019의 비판 기록, 충실한 재현 위해 유지) → PCA (k=6) → 유한차분 `Ẋ` → `M = -Mᵀ` 제약 하에서 `Ẋ = M X` 적합 (반대칭 제약 최소제곱, 닫힌 형태) → 고유분해 (±iω) → 최상위 평면.
**합격/불합격:** `Ẋ=MX`의 적합 R²; 최상위 회전 평면 분산 비율; Antin `jPCA` 대비 주각(principal angle) < 5°, 주파수 오차 5% 이내.

In [ ]:
# TODO(M0): jPCA hand implementation + Antin differential test. (Do not start until approved.)
# TODO(M0): jPCA 직접 구현 + Antin 차등 검증. (승인 전까지 시작하지 않음.)

## M1 · Flip-flop calibration (known answer — DO FIRST)
**(1) math/why:** train a small tanh RNN on the 3-bit flip-flop; run the fixed-point finder.
**FALSIFICATION GATE:** exactly **8 stable fixed points** near cube corners, else STOP and debug before touching reaching data. Differential test vs `pytorch-fixed-point-analysis`. Render 8 FPs + trajectories in the minimal viewer.

---

## M1 · 플립플롭 보정 (정답이 알려진 사례 — 가장 먼저)
**(1) 수식/이유:** 작은 tanh RNN을 3-bit 플립플롭 과제로 학습 → 고정점 탐색기 실행.
**반증 게이트:** 큐브 꼭짓점 근처에 **정확히 8개의 안정 고정점**. 아니면 도달(reaching) 데이터를 건드리기 전에 멈추고 탐색기를 디버그한다. `pytorch-fixed-point-analysis`와 차등 검증. 최소 뷰어에 8개 고정점 + 궤적 렌더링.

In [ ]:
# TODO(M1): flip-flop RNN + fixed-point finder + 8-corner gate + minimal viewer.
# TODO(M1): 플립플롭 RNN + 고정점 탐색기 + 8-꼭짓점 게이트 + 최소 뷰어.

## M2 · Brain data — PCA + jPCA (Stage 1)
**(1) math/why:** MC_Maze via nlb_tools, 20 ms bins, Gaussian σ=40 ms, soft-norm, aligned on movement onset (prep + movement). Apply the M0 pipeline.
**Adversarial:** does the rotation survive *without* cross-condition-mean subtraction? Report before/after. Brain viewer: trajectories + jPCA plane, **no fixed points**.

---

## M2 · 뇌 데이터 — PCA + jPCA (1단계)
**(1) 수식/이유:** nlb_tools로 MC_Maze, 20 ms 빈, 가우시안 σ=40 ms, 소프트 정규화, 움직임 개시(movement onset) 기준 정렬 (준비 + 움직임 구간). M0 파이프라인 적용.
**적대적 검증:** 조건간 평균 차감을 *하지 않아도* 회전이 남는가? 전/후를 보고한다. 뇌 뷰어: 궤적 + jPCA 평면, **고정점 없음**.

In [ ]:
# TODO(M2): MC_Maze load/preprocess + jPCA on brain data + adversarial CCM check + brain viewer.
# TODO(M2): MC_Maze 로드/전처리 + 뇌 데이터 jPCA + 조건간평균(CCM) 적대적 검증 + 뇌 뷰어.

## M3 · Reaching RNN + jPCA (Stage 2 part 1)
**(1) math/why:** 256-unit continuous-time tanh RNN, `dx/dt = -x + W_rec·φ(x) + W_in·u + b`, `z = W_out·x`; inputs = target/go, output = hand velocity; **metabolic L2-on-rates mandatory**. Trained on the TASK, **not** on spikes.
**Task gate:** report velocity R², require above threshold before any dynamics analysis. Then apply PCA/jPCA to hidden states — does the RNN rotate?

---

## M3 · 도달(reaching) RNN + jPCA (2단계 1부)
**(1) 수식/이유:** 256유닛 연속시간 tanh RNN, `dx/dt = -x + W_rec·φ(x) + W_in·u + b`, `z = W_out·x`; 입력 = 목표/go 신호, 출력 = 손 속도; **발화율 L2 대사(metabolic) 정규화 필수**. **스파이크가 아니라 과제(TASK)**로 학습한다.
**과제 게이트:** 속도 R²를 보고하고, 동역학 분석 전에 임계값 이상일 것을 요구한다. 그다음 은닉 상태에 PCA/jPCA 적용 — RNN도 회전하는가?

In [ ]:
# TODO(M3): reaching RNN (task-trained) + velocity R² gate + jPCA on hidden states.
# TODO(M3): 도달 RNN (과제 학습) + 속도 R² 게이트 + 은닉 상태 jPCA.

## M4 · Reaching RNN fixed points + flow field (Stage 2 part 2)
**(1) math/why:** minimize `q(x)=½‖dx/dt‖²` (Adam, optional L-BFGS polish); ICs sampled **only from states on real trials + noise**; tolerance from the q-**distribution**, not a hard-coded absolute; de-dup by clustering; classify by Jacobian eigenvalues (stable/unstable/saddle/rotational); `dx/dt` on a grid in the top PCA plane.
**Adversarial:** FPs stable to IC re-sampling? Hand vs reference agree after Hungarian alignment? RNN viewer: trajectories + flow + FPs by class.

---

## M4 · 도달 RNN 고정점 + 흐름장 (2단계 2부)
**(1) 수식/이유:** `q(x)=½‖dx/dt‖²` 최소화 (Adam, 선택적 L-BFGS 다듬기); 초기조건(IC)은 **실제 시행에서 방문한 상태 + 노이즈에서만** 샘플링; 허용오차는 절대값 하드코딩이 아니라 q **분포**에서 결정; 군집화로 중복 제거; 야코비안 고유값으로 분류 (안정/불안정/안장/회전); 최상위 PCA 평면 격자에서 `dx/dt` 평가.
**적대적 검증:** IC 재샘플링에도 고정점이 안정적인가? 헝가리안 정렬 후 직접 구현 vs 기준이 일치하는가? RNN 뷰어: 궤적 + 흐름장 + 안정성 분류별 고정점.

In [ ]:
# TODO(M4): fixed/slow point finder + Jacobian classification + flow field + RNN viewer + adversarial checks.
# TODO(M4): 고정점/저속점 탐색기 + 야코비안 분류 + 흐름장 + RNN 뷰어 + 적대적 검증.

## M5 · Unified viewer + deploy
**(1) math/why:** toggle Brain | RNN | side-by-side. The two live in **different spaces** — render each in its own jPCA space, match only visual scale, surface the limitation honestly. Export JSON via `export.py`; deploy the static app to Hugging Face Spaces (Static SDK).

---

## M5 · 통합 뷰어 + 배포
**(1) 수식/이유:** 뇌 | RNN | 나란히 보기 토글. 둘은 **서로 다른 공간**에 존재한다 — 각자의 jPCA 공간에서 렌더링하고 시각적 스케일만 맞추며, 이 한계를 정직하게 드러낸다. `export.py`로 JSON 내보내기; 정적 앱을 Hugging Face Spaces(Static SDK)에 배포.

In [ ]:
# TODO(M5): export JSON + unified viewer + deploy.
# TODO(M5): JSON 내보내기 + 통합 뷰어 + 배포.